In [1]:
import ROOT
import math
import os
import pandas as pd
import numpy as np

%jsroot on

# ============================================================
# USER SETTINGS
# ============================================================

file_path = "/root/geant4/detector/Lung_Elements/HCO.root"
tree_name = "t"

selected_volume = 5
selected_pdg = 22
selected_track = 1

E0 = 662.0                 # incident gamma energy, keV
ME = 511.0                 # electron rest energy, keV
P_AU = 3.72738             # 1 a.u. momentum = 3.72738 keV/c

# Incident gamma direction
# /gps/direction 0 -1 0
incident_direction = (0.0, -1.0, 0.0)

# Output CSV
csv_file = "HCO_vlm5_trk1_all_records_Eprime_theta_Q.csv"


# ============================================================
# OPEN ROOT FILE
# ============================================================

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"ROOT file not found:\n{file_path}"
    )

root_file = ROOT.TFile.Open(file_path)

if not root_file or root_file.IsZombie():
    raise RuntimeError(
        f"Could not open ROOT file:\n{file_path}"
    )

tree = root_file.Get(tree_name)

if not tree:
    root_file.ls()
    raise RuntimeError(
        f"TTree '{tree_name}' was not found."
    )

print("========================================")
print("ROOT FILE")
print("========================================")
print("File:", file_path)
print("Tree:", tree_name)
print("Tree entries:", tree.GetEntries())


# ============================================================
# CHECK REQUIRED BRANCHES
# ============================================================

required_branches = [
    "pdg",
    "vlm",
    "trk",
    "stp",
    "k",
    "px",
    "py",
    "pz"
]

available_branches = {
    branch.GetName()
    for branch in tree.GetListOfBranches()
}

missing_branches = [
    name
    for name in required_branches
    if name not in available_branches
]

if missing_branches:
    root_file.Close()

    raise RuntimeError(
        "Missing required branches: "
        + ", ".join(missing_branches)
    )

print("\nAll required branches are available.")


# ============================================================
# NORMALIZE INCIDENT DIRECTION
# ============================================================

ix, iy, iz = incident_direction

incident_norm = math.sqrt(
    ix**2 + iy**2 + iz**2
)

if incident_norm <= 0:
    root_file.Close()
    raise ValueError(
        "Incident direction cannot be zero."
    )

ix /= incident_norm
iy /= incident_norm
iz /= incident_norm

print(
    "Incident direction:",
    ix,
    iy,
    iz
)


# ============================================================
# STORAGE
# ============================================================

rows = []

root_like_count = 0

invalid_energy = 0
invalid_momentum = 0
negative_q2 = 0
zero_q_count = 0


# ============================================================
# EVENT LOOP
# ============================================================

for event_number, event in enumerate(tree):

    n_core = min(
        len(event.pdg),
        len(event.vlm),
        len(event.trk),
        len(event.stp),
        len(event.k),
        len(event.px),
        len(event.py),
        len(event.pz)
    )

    for i in range(n_core):

        # ====================================================
        # EXACT ROOT-LIKE SELECTION
        #
        # vlm == 5 && pdg == 22 && trk == 1
        # ====================================================

        if int(event.vlm[i]) != selected_volume:
            continue

        if int(event.pdg[i]) != selected_pdg:
            continue

        if int(event.trk[i]) != selected_track:
            continue

        root_like_count += 1


        # ====================================================
        # E' = k
        # ====================================================

        Eprime_keV = float(event.k[i])

        if not math.isfinite(Eprime_keV):
            invalid_energy += 1
            continue

        if Eprime_keV < 0.0:
            invalid_energy += 1
            continue


        # ====================================================
        # ENERGY LOSS
        #
        # Delta E = E0 - E'
        # ====================================================

        E_loss_keV = E0 - Eprime_keV


        # ====================================================
        # px, py, pz
        # ====================================================

        px = float(event.px[i])
        py = float(event.py[i])
        pz = float(event.pz[i])

        if not (
            math.isfinite(px)
            and math.isfinite(py)
            and math.isfinite(pz)
        ):
            invalid_momentum += 1
            continue


        # ====================================================
        # OUTGOING MOMENTUM MAGNITUDE
        # ====================================================

        p_mag = math.sqrt(
            px**2 +
            py**2 +
            pz**2
        )

        # ----------------------------------------------------
        # If momentum is zero, theta cannot be calculated.
        # Keep the row with NaN theta/Q.
        # ----------------------------------------------------

        if p_mag <= 0.0:

            invalid_momentum += 1

            theta_deg = float("nan")
            cos_theta = float("nan")
            q_keV_c = float("nan")

            Q_keV_c = float("nan")
            Q_au = float("nan")
            Q_abs_au = float("nan")

        else:

            # =================================================
            # NORMALIZED OUTGOING DIRECTION
            # =================================================

            ux = px / p_mag
            uy = py / p_mag
            uz = pz / p_mag


            # =================================================
            # SCATTERING ANGLE
            #
            # theta = angle between incident and outgoing gamma
            # =================================================

            cos_theta = (
                ix * ux +
                iy * uy +
                iz * uz
            )

            cos_theta = max(
                -1.0,
                min(1.0, cos_theta)
            )

            theta_rad = math.acos(
                cos_theta
            )

            theta_deg = math.degrees(
                theta_rad
            )


            # =================================================
            # PHOTON MOMENTUM TRANSFER q
            #
            # q^2 = E0^2 + E'^2
            #       - 2 E0 E' cos(theta)
            #
            # q in keV/c
            # =================================================

            q_squared = (
                E0**2
                + Eprime_keV**2
                - 2.0
                * E0
                * Eprime_keV
                * cos_theta
            )


            # =================================================
            # HANDLE q
            # =================================================

            if not math.isfinite(q_squared):

                q_keV_c = float("nan")
                Q_keV_c = float("nan")
                Q_au = float("nan")
                Q_abs_au = float("nan")

            elif q_squared < 0.0:

                # Small negative values can occur from
                # numerical roundoff.
                if abs(q_squared) < 1.0e-8:

                    q_squared = 0.0

                else:

                    negative_q2 += 1

                    q_keV_c = float("nan")
                    Q_keV_c = float("nan")
                    Q_au = float("nan")
                    Q_abs_au = float("nan")

                    rows.append({
                        "event_number": event_number,
                        "record_index": i,
                        "step_number": int(event.stp[i]),

                        "vlm": int(event.vlm[i]),
                        "pdg": int(event.pdg[i]),
                        "track_id": int(event.trk[i]),

                        "E0_keV": E0,
                        "Eprime_k_keV": Eprime_keV,
                        "E0_minus_Eprime_keV": E_loss_keV,

                        "px": px,
                        "py": py,
                        "pz": pz,

                        "theta_deg": theta_deg,
                        "cos_theta": cos_theta,

                        "q_gamma_keV_c": q_keV_c,

                        "Q_keV_c": Q_keV_c,
                        "Q_au": Q_au,
                        "Q_abs_au": Q_abs_au
                    })

                    continue


            # =================================================
            # q >= 0
            # =================================================

            q_keV_c = math.sqrt(
                max(q_squared, 0.0)
            )


            # =================================================
            # q = 0
            #
            # This is typically:
            #
            # E' = E0
            # theta = 0
            #
            # Q formula contains ME/q,
            # so Q is undefined.
            #
            # KEEP THE EVENT.
            # =================================================

            if q_keV_c < 1.0e-10:

                zero_q_count += 1

                Q_keV_c = float("nan")
                Q_au = float("nan")
                Q_abs_au = float("nan")


            # =================================================
            # q > 0: CALCULATE Q
            # =================================================

            else:

                Q_keV_c = (
                    ME / q_keV_c
                ) * (
                    E_loss_keV
                    - q_squared / (2.0 * ME)
                )

                Q_au = (
                    Q_keV_c / P_AU
                )

                Q_abs_au = abs(
                    Q_au
                )


        # ====================================================
        # SAVE EVERY ROOT-LIKE RECORD
        # ====================================================

        rows.append({

            "event_number":
                event_number,

            "record_index":
                i,

            "step_number":
                int(event.stp[i]),

            "vlm":
                int(event.vlm[i]),

            "pdg":
                int(event.pdg[i]),

            "track_id":
                int(event.trk[i]),

            "E0_keV":
                E0,

            "Eprime_k_keV":
                Eprime_keV,

            "E0_minus_Eprime_keV":
                E_loss_keV,

            "px":
                px,

            "py":
                py,

            "pz":
                pz,

            "theta_deg":
                theta_deg,

            "cos_theta":
                cos_theta,

            "q_gamma_keV_c":
                q_keV_c,

            "Q_keV_c":
                Q_keV_c,

            "Q_au":
                Q_au,

            "Q_abs_au":
                Q_abs_au
        })


# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(rows)


# ============================================================
# CHECK RESULTS
# ============================================================

print("\n========================================")
print("SELECTION SUMMARY")
print("========================================")

print(
    f"Selection: vlm=={selected_volume} "
    f"&& pdg=={selected_pdg} "
    f"&& trk=={selected_track}"
)

print(
    "ROOT-like matching records:",
    root_like_count
)

print(
    "Rows saved to dataframe:",
    len(df)
)

print(
    "Invalid energy:",
    invalid_energy
)

print(
    "Zero/invalid momentum:",
    invalid_momentum
)

print(
    "Negative q^2:",
    negative_q2
)

print(
    "q = 0 records:",
    zero_q_count
)


if df.empty:

    root_file.Close()

    raise RuntimeError(
        "No records were saved."
    )


# ============================================================
# DATA RANGES
# ============================================================

print("\n========================================")
print("ENERGY RESULTS")
print("========================================")

print(
    "Minimum Eprime:",
    df["Eprime_k_keV"].min(),
    "keV"
)

print(
    "Maximum Eprime:",
    df["Eprime_k_keV"].max(),
    "keV"
)

print(
    "Mean Eprime:",
    df["Eprime_k_keV"].mean(),
    "keV"
)

print(
    "Median Eprime:",
    df["Eprime_k_keV"].median(),
    "keV"
)


# ============================================================
# COUNT NEAR 662 keV
# ============================================================

near_662 = df[
    np.abs(
        df["Eprime_k_keV"] - E0
    ) < 0.5
]

print(
    "\nRecords with Eprime within "
    "0.5 keV of 662 keV:",
    len(near_662)
)


# ============================================================
# PRINT THETA RANGE
# ============================================================

valid_theta_check = df[
    np.isfinite(
        df["theta_deg"]
    )
]

if not valid_theta_check.empty:

    print(
        "\nTheta range:",
        valid_theta_check["theta_deg"].min(),
        "to",
        valid_theta_check["theta_deg"].max(),
        "degrees"
    )


# ============================================================
# PRINT Q RANGE
# ============================================================

valid_Q_check = df[
    np.isfinite(
        df["Q_au"]
    )
]

if not valid_Q_check.empty:

    print(
        "Signed Q range:",
        valid_Q_check["Q_au"].min(),
        "to",
        valid_Q_check["Q_au"].max(),
        "a.u."
    )

    print(
        "|Q| range:",
        valid_Q_check["Q_abs_au"].min(),
        "to",
        valid_Q_check["Q_abs_au"].max(),
        "a.u."
    )


# ============================================================
# SAVE COMPLETE CSV
# ============================================================

df_save = df.copy()

df_save = df_save.round(8)

df_save.to_csv(
    csv_file,
    index=False
)

print("\n========================================")
print("CSV SAVED")
print("========================================")

print(
    os.path.abspath(csv_file)
)


# ============================================================
# IMPORTANT:
#
# df contains ALL records including q=0.
#
# For theta plot:
# use rows with valid theta.
#
# For Q plots:
# use only rows where Q is defined.
# ============================================================


# ============================================================
# DATA FOR THETA PLOT
# ============================================================

df_theta = df[
    np.isfinite(
        df["theta_deg"]
    )
    &
    np.isfinite(
        df["E0_minus_Eprime_keV"]
    )
].copy()


# ============================================================
# DATA FOR Q PLOTS
# ============================================================

df_Q = df[
    np.isfinite(
        df["Q_au"]
    )
    &
    np.isfinite(
        df["Q_abs_au"]
    )
    &
    np.isfinite(
        df["theta_deg"]
    )
    &
    np.isfinite(
        df["E0_minus_Eprime_keV"]
    )
].copy()


print("\n========================================")
print("PLOT DATA")
print("========================================")

print(
    "Points in theta plot:",
    len(df_theta)
)

print(
    "Points in Q plots:",
    len(df_Q)
)


# ============================================================
# NUMPY ARRAYS
# ============================================================

theta_array = (
    df_theta["theta_deg"]
    .to_numpy(dtype=np.float64)
)

Eloss_theta_array = (
    df_theta["E0_minus_Eprime_keV"]
    .to_numpy(dtype=np.float64)
)


Q_array = (
    df_Q["Q_au"]
    .to_numpy(dtype=np.float64)
)

Qabs_array = (
    df_Q["Q_abs_au"]
    .to_numpy(dtype=np.float64)
)

theta_Q_array = (
    df_Q["theta_deg"]
    .to_numpy(dtype=np.float64)
)

Eloss_Q_array = (
    df_Q["E0_minus_Eprime_keV"]
    .to_numpy(dtype=np.float64)
)


# ============================================================
# REMOVE OLD ROOT OBJECTS
# ============================================================

object_names = [
    "g_theta_Eloss",
    "g_Q_Eloss",
    "g_Qabs_Eloss",
    "g3_theta_Qabs_Eloss",
    "c_HCO_analysis"
]

for name in object_names:

    obj = ROOT.gROOT.FindObject(
        name
    )

    if obj:

        if obj.InheritsFrom(
            "TCanvas"
        ):
            obj.Close()

        else:
            obj.Delete()


# ============================================================
# GRAPH 1
#
# theta vs (E0-Eprime)
#
# X = theta
# Y = E0-Eprime
# ============================================================

g_theta_Eloss = ROOT.TGraph(
    len(theta_array),
    theta_array,
    Eloss_theta_array
)

g_theta_Eloss.SetName(
    "g_theta_Eloss"
)

g_theta_Eloss.SetTitle(
    "Volume 5: #theta vs E_{0}-E';"
    "#theta (degrees);"
    "E_{0}-E' (keV)"
)

g_theta_Eloss.SetMarkerStyle(20)
g_theta_Eloss.SetMarkerSize(0.30)


# ============================================================
# GRAPH 2
#
# signed Q vs (E0-Eprime)
#
# X = Q
# Y = E0-Eprime
# ============================================================

g_Q_Eloss = ROOT.TGraph(
    len(Q_array),
    Q_array,
    Eloss_Q_array
)

g_Q_Eloss.SetName(
    "g_Q_Eloss"
)

g_Q_Eloss.SetTitle(
    "Volume 5: signed Q vs E_{0}-E';"
    "Signed Q (a.u.);"
    "E_{0}-E' (keV)"
)

g_Q_Eloss.SetMarkerStyle(20)
g_Q_Eloss.SetMarkerSize(0.30)


# ============================================================
# GRAPH 3
#
# |Q| vs (E0-Eprime)
#
# X = |Q|
# Y = E0-Eprime
# ============================================================

g_Qabs_Eloss = ROOT.TGraph(
    len(Qabs_array),
    Qabs_array,
    Eloss_Q_array
)

g_Qabs_Eloss.SetName(
    "g_Qabs_Eloss"
)

g_Qabs_Eloss.SetTitle(
    "Volume 5: |Q| vs E_{0}-E';"
    "|Q| (a.u.);"
    "E_{0}-E' (keV)"
)

g_Qabs_Eloss.SetMarkerStyle(20)
g_Qabs_Eloss.SetMarkerSize(0.30)


# ============================================================
# GRAPH 4
#
# 3D:
#
# X = theta
# Y = |Q|
# Z = E0-Eprime
# ============================================================

g3_theta_Qabs_Eloss = ROOT.TGraph2D(
    len(theta_Q_array),
    theta_Q_array,
    Qabs_array,
    Eloss_Q_array
)

g3_theta_Qabs_Eloss.SetName(
    "g3_theta_Qabs_Eloss"
)

g3_theta_Qabs_Eloss.SetTitle(
    "Volume 5: #theta vs |Q| vs E_{0}-E';"
    "#theta (degrees);"
    "|Q| (a.u.);"
    "E_{0}-E' (keV)"
)

g3_theta_Qabs_Eloss.SetMarkerStyle(20)
g3_theta_Qabs_Eloss.SetMarkerSize(0.30)


# ============================================================
# CREATE CANVAS
# ============================================================

c_HCO_analysis = ROOT.TCanvas(
    "c_HCO_analysis",
    "HCO volume 5 analysis",
    1700,
    1200
)

c_HCO_analysis.Divide(
    2,
    2
)


# ============================================================
# PLOT 1
# theta vs E0-Eprime
# ============================================================

pad1 = c_HCO_analysis.cd(1)

pad1.SetLeftMargin(0.12)
pad1.SetBottomMargin(0.12)
pad1.SetGrid()

g_theta_Eloss.Draw("AP")

g_theta_Eloss.GetXaxis().SetLimits(
    0.0,
    180.0
)

g_theta_Eloss.GetXaxis().SetTitle(
    "#theta (degrees)"
)

g_theta_Eloss.GetYaxis().SetTitle(
    "E_{0}-E' (keV)"
)


# ============================================================
# PLOT 2
# signed Q vs E0-Eprime
# ============================================================

pad2 = c_HCO_analysis.cd(2)

pad2.SetLeftMargin(0.12)
pad2.SetBottomMargin(0.12)
pad2.SetGrid()

g_Q_Eloss.Draw("AP")

g_Q_Eloss.GetXaxis().SetTitle(
    "Signed Q (a.u.)"
)

g_Q_Eloss.GetYaxis().SetTitle(
    "E_{0}-E' (keV)"
)


# ============================================================
# PLOT 3
# |Q| vs E0-Eprime
# ============================================================

pad3 = c_HCO_analysis.cd(3)

pad3.SetLeftMargin(0.12)
pad3.SetBottomMargin(0.12)
pad3.SetGrid()

g_Qabs_Eloss.Draw("AP")

g_Qabs_Eloss.GetXaxis().SetTitle(
    "|Q| (a.u.)"
)

g_Qabs_Eloss.GetYaxis().SetTitle(
    "E_{0}-E' (keV)"
)


# ============================================================
# PLOT 4
# 3D theta vs |Q| vs E0-Eprime
# ============================================================

pad4 = c_HCO_analysis.cd(4)

pad4.SetLeftMargin(0.12)
pad4.SetRightMargin(0.15)
pad4.SetBottomMargin(0.12)

g3_theta_Qabs_Eloss.Draw("P0")

g3_theta_Qabs_Eloss.GetXaxis().SetTitle(
    "#theta (degrees)"
)

g3_theta_Qabs_Eloss.GetYaxis().SetTitle(
    "|Q| (a.u.)"
)

g3_theta_Qabs_Eloss.GetZaxis().SetTitle(
    "E_{0}-E' (keV)"
)


# ============================================================
# DISPLAY
# ============================================================

c_HCO_analysis.Modified()
c_HCO_analysis.Update()
c_HCO_analysis.Draw()


# ============================================================
# CLOSE ROOT FILE
# ============================================================

root_file.Close()

print("\n========================================")
print("ANALYSIS COMPLETED")
print("========================================")

ROOT FILE
File: /root/geant4/detector/Lung_Elements/HCO.root
Tree: t
Tree entries: 100000

All required branches are available.
Incident direction: 0.0 -1.0 0.0

SELECTION SUMMARY
Selection: vlm==5 && pdg==22 && trk==1
ROOT-like matching records: 1417
Rows saved to dataframe: 1417
Invalid energy: 0
Zero/invalid momentum: 0
Negative q^2: 0
q = 0 records: 0

ENERGY RESULTS
Minimum Eprime: 0.0 keV
Maximum Eprime: 660.7227074997903 keV
Mean Eprime: 468.16806174458696 keV
Median Eprime: 552.4412497246368 keV

Records with Eprime within 0.5 keV of 662 keV: 0

Theta range: 0.5746260983111763 to 173.95512308919928 degrees
Signed Q range: -59.01384092118737 to 122.00565878847759 a.u.
|Q| range: 0.0060431560958243815 to 122.00565878847759 a.u.

CSV SAVED
/root/geant4/detector/Lung_Elements/HCO_vlm5_trk1_all_records_Eprime_theta_Q.csv

PLOT DATA
Points in theta plot: 1417
Points in Q plots: 1417

ANALYSIS COMPLETED
